## Step 0: Import Necessary Libraries

In [ ]:
import ee
import requests
import io
import numpy as np
import os
import torch
from torch import Tensor, nn
import copy
import numpy as np
import torch
import os
from tqdm import tqdm

## Step 1: Preparing RS/MO/DEM Data (GEE)

In [ ]:
# BERTH's Default Band Order
columns_mo = ['ws_u_10', 'ws_v_10', 't_dewpoint', 't_air', 't_skin','pressure', 'down_radiation_short', 'down_radiation_long']
columns_rs = ['blue', 'green', 'red', 'nir', 'swir1', 'swir2']
columns_terrain = ['dem', 'aspect', 'slope']
columns_hydro = ['precipitation', 'soilmoisture', 'evapotranspiration', 'runoff']
columns_berth = columns_rs + columns_mo + columns_terrain

# GEE
ee.Authenticate()
ee.Initialize(project='ee-zhaoyuan-yao')

# Export Setting
region = {"west": 38.2378, "north": 30.1577, "south": 30.0446, "east": 38.4369}
region_name = 'Example_Hail_Arabia'
region_ee = ee.Geometry.BBox(region['west'], region['south'], region['east'], region['north'])
TIME_START = ee.Date('2017-01-01')
TIME_END = ee.Date('2020-12-31')
TIME_LENGTH = TIME_END.difference(TIME_START, 'day').getInfo() + 1

In [ ]:
# Download DEM via GEE
def preprocess_terrain():
    dem = ee.ImageCollection('COPERNICUS/DEM/GLO30').select('DEM').mosaic().setDefaultProjection('EPSG:3857', None, 30).resample('bilinear').rename('dem')
    slope = ee.Terrain.slope(dem).rename('slope')
    aspect = ee.Terrain.aspect(dem).rename('aspect')
    return dem.unmask(-1000).rename('dem').addBands(aspect.unmask(-1)).addBands(slope.unmask(-1))

url = ee.Image(preprocess_terrain()).select(columns_terrain).float().getDownloadURL({"region": region_ee, "scale": 30, "crs": "EPSG:4326", "format": "NPY"})
response = requests.get(url)
response.raise_for_status()
terrain_data = np.load(io.BytesIO(response.content), allow_pickle=True)
np.save(f'{region_name}_30m_Terrain_GLO30.npy', terrain_data.view((np.float32, len(columns_terrain))))
print(terrain_data.shape)

In [ ]:
# Download Atmospheric Forcing via GEE
def preprocess_era5land(date_offset):
    date = TIME_START.advance(date_offset, 'day')
    date_filter = ee.Filter.date(date.advance(-0.1, 'day'), date.advance(1, 'day'))
    mo_image = ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR').filter(date_filter).first()
    ws_u_10 = mo_image.select('u_component_of_wind_10m').divide(20.0)
    ws_v_10 = mo_image.select('v_component_of_wind_10m').divide(20.0)
    t_dewpoint = mo_image.select('dewpoint_temperature_2m').subtract(200).divide(150.0)
    t_air = mo_image.select('temperature_2m').subtract(200).divide(150.0)
    t_skin = mo_image.select('skin_temperature').subtract(200).divide(150.0)
    p = mo_image.select('surface_pressure').divide(1000).subtract(70).divide(80.0)
    down_short = mo_image.select('surface_solar_radiation_downwards_sum').multiply(1.1574E-5).divide(1000)
    down_long  = mo_image.select('surface_thermal_radiation_downwards_sum').multiply(1.1574E-5).divide(1000)
    data = ws_u_10.rename('ws_u_10')\
        .addBands(ws_v_10.rename('ws_v_10'))\
        .addBands(t_dewpoint.rename('t_dewpoint'))\
        .addBands(t_air.rename('t_air'))\
        .addBands(t_skin.rename('t_skin'))\
        .addBands(p.rename('pressure'))\
        .addBands(down_short.rename('down_radiation_short'))\
        .addBands(down_long.rename('down_radiation_long'))
    return data.resample('bilinear')
if not os.path.exists(f'{region_name}_30m_MO'):
    os.mkdir(f'{region_name}_30m_MO')
for day_offset in range(TIME_LENGTH):
    url = ee.Image(preprocess_era5land(day_offset)).select(columns_mo).float().getDownloadURL({"region": region_ee, "scale": 30, "crs": "EPSG:4326", "format": "NPY"})
    response = requests.get(url, stream=True)
    response.raise_for_status()
    mo_data = np.load(io.BytesIO(response.content), allow_pickle=True)
    np.save(f'{region_name}_30m_MO/{region_name}_30m_MO_{day_offset}.npy', mo_data.view((np.float32, len(columns_mo))))
    print(f'day:{day_offset}, MO, {mo_data.shape}')

In [ ]:
# Download Surface Reflectance via GEE
def preprocess_single_LS89(image):
    dilatedCloudBitMask = (1 << 1)
    cirrusBitMask = (1 << 2)
    cloudBitMask = (1 << 3)
    cloudShadowBitMask = (1 << 4)
    qa = image.select('QA_PIXEL')
    mask = qa.bitwiseAnd(cloudShadowBitMask).eq(0).And(qa.bitwiseAnd(cirrusBitMask).eq(0)).And(qa.bitwiseAnd(dilatedCloudBitMask).eq(0)).And(qa.bitwiseAnd(cloudBitMask).eq(0))
    red  = ee.Image(image).select('SR_B4').multiply(2.75e-05).add(-0.2)
    nir = ee.Image(image).select('SR_B5').multiply(2.75e-05).add(-0.2)
    blue  = ee.Image(image).select('SR_B2').multiply(2.75e-05).add(-0.2)
    green = ee.Image(image).select('SR_B3').multiply(2.75e-05).add(-0.2)
    swir1 = ee.Image(image).select('SR_B6').multiply(2.75e-05).add(-0.2)
    swir2 = ee.Image(image).select('SR_B7').multiply(2.75e-05).add(-0.2)
    data = red.rename('red')\
        .addBands(blue.rename('blue'))\
        .addBands(green.rename('green'))\
        .addBands(nir.rename('nir'))\
        .addBands(swir1.rename('swir1'))\
        .addBands(swir2.rename('swir2'))\
        .updateMask(mask)
    return data.float()
def preprocess_single_S2(image):
    qa = image.select('QA60')
    scl = image.select('SCL')
    mask = qa.bitwiseAnd(1 << 10).eq(0).And(qa.bitwiseAnd(1 << 11).eq(0)).And(scl.neq(3)).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))
    red  = ee.Image(image).select('B4').multiply(0.0001).multiply(0.982).add(0.00094)
    nir = ee.Image(image).select('B8').multiply(0.0001).multiply(1.001).add(-0.00029)
    blue  = ee.Image(image).select('B2').multiply(0.0001).multiply(0.977).add(-0.00411)
    green = ee.Image(image).select('B3').multiply(0.0001).multiply(1.005).add(-0.00093)
    swir1 = ee.Image(image).select('B11').multiply(0.0001).multiply(1.001).add(-0.00015)
    swir2 = ee.Image(image).select('B12').multiply(0.0001).multiply(0.996).add(-0.00097)
    data = red.rename('red')\
        .addBands(blue.rename('blue'))\
        .addBands(green.rename('green'))\
        .addBands(nir.rename('nir'))\
        .addBands(swir1.rename('swir1'))\
        .addBands(swir2.rename('swir2'))\
        .updateMask(mask)
    return data.float()
def preprocess_remotesensing(date_offset):
    date = TIME_START.advance(date_offset, 'day')
    date_filter = ee.Filter.date(date, date.advance(1, 'day'))
    day_images_ls89 = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').filter(date_filter).merge(ee.ImageCollection('LANDSAT/LC09/C02/T1_L2').filter(date_filter)).filter(ee.Filter.equals('PROCESSING_LEVEL', 'L2SP')).map(preprocess_single_LS89)
    day_images_s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filter(date_filter)\
        .filter(ee.Filter.neq('system:index', '20190417T065631_20190417T070736_T38NRL'))\
        .filter(ee.Filter.neq('system:index', '20190419T105621_20190419T105622_T41XML'))\
        .filter(ee.Filter.neq('system:index', '20190119T210811_20190119T210805_T06VXK'))\
        .filter(ee.Filter.neq('system:index', '20190117T010959_20190117T011000_T55TEN'))\
        .filter(ee.Filter.neq('system:index', '20190117T061209_20190117T061411_T42TVK'))\
        .filter(ee.Filter.neq('system:index', '20190117T140051_20190117T141300_T20HND'))\
        .map(preprocess_single_S2)
    day_images = ee.ImageCollection(day_images_ls89).merge(day_images_s2)
    data = ee.Algorithms.If(condition=ee.ImageCollection(day_images).first(),\
                            trueCase=day_images.mean().unmask(-1),\
                            falseCase=ee.Image.constant([-1.0, -1.0, -1.0, -1.0, -1.0, -1.0]).rename(['red', 'blue', 'green', 'nir', 'swir1', 'swir2']))
    return ee.Image(data).float()
if not os.path.exists(f'{region_name}_30m_RS'):
    os.mkdir(f'{region_name}_30m_RS')
for day_offset in range(TIME_LENGTH):
    url = ee.Image(preprocess_remotesensing(day_offset)).select(columns_rs).float().getDownloadURL({"region": region_ee, "scale": 30, "crs": "EPSG:4326", "format": "NPY"})
    response = requests.get(url, stream=True)
    response.raise_for_status()
    rs_data = np.load(io.BytesIO(response.content), allow_pickle=True)
    np.save(f'{region_name}_30m_RS/{region_name}_30m_RS_{day_offset}.npy', rs_data.view((np.float32, len(columns_rs))))
    print(f'day:{day_offset}, RS, {rs_data.shape}')

## Step-2: Run BERTH

In [ ]:
# Define BERTH
class HydroTrans(nn.Module):
    def __init__(self, rs_dim=6, mo_dim=7, out_dim=4, max_len=500):
        super().__init__()
        self.model_hidden = 256
        self.model_head = 4
        self.model_layer = 6
        single_encoder_layer = nn.TransformerEncoderLayer(d_model=self.model_hidden, nhead=self.model_head,
                                                          dropout=0, batch_first=True)
        combined_encoder_layer = nn.TransformerEncoderLayer(d_model=self.model_hidden * 3, nhead=self.model_head,
                                                            dropout=0, batch_first=True)
        self.rs_embedding = nn.Linear(rs_dim, self.model_hidden)
        self.rs_encoder = nn.TransformerEncoder(single_encoder_layer, num_layers=3)
        self.mo_embedding = nn.Linear(mo_dim, self.model_hidden)
        self.mo_encoder = nn.TransformerEncoder(single_encoder_layer, num_layers=3)
        self.topo_embedding = nn.Linear(3, self.model_hidden)
        self.combined_encoder = nn.TransformerEncoder(combined_encoder_layer, num_layers=3)
        self.position_embedding = nn.Embedding(max_len, self.model_hidden)
        self.linear = nn.Linear(self.model_hidden * 3, out_dim)

        for param in self.parameters():
            param.requires_grad = True
        self.max_length = max_len

    def forward(self, rs, mo, topo):
        position = torch.arange(self.max_length).unsqueeze(0).to(rs.device)
        p_eb = self.position_embedding(position)
        rs_features = self.rs_encoder(self.rs_embedding(rs) + p_eb, src_key_padding_mask=rs[:, :, 0] == -1)
        mo_features = self.mo_encoder(self.mo_embedding(mo) + p_eb)
        topo_features = self.topo_embedding(topo).unsqueeze(1).repeat(1, 500, 1)

        combined_features = torch.concat([rs_features, mo_features, topo_features], dim=2)

        hydro_features = self.combined_encoder.forward(combined_features)
        return self.linear(hydro_features)

# Load BERTH
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model = HydroTrans(rs_dim=6, mo_dim=7, out_dim=4, max_len=500)
model.load_state_dict(torch.load('BERTH_LS789S2_v1.pt', map_location='cpu'))
model.to(device)
model.eval()

In [ ]:
# Load Data
grid_topo = np.load(f'{region_name}_30m_Terrain_GLO30.npy')
grid_x = grid_topo.shape[0]
grid_y = grid_topo.shape[1]
grid_topo[:, :, 0] = grid_topo[:, :, 0] / 4000
grid_topo[:, :, 1] = grid_topo[:, :, 1] / 360
grid_topo[:, :, 2] = grid_topo[:, :, 2] / 30
grid_mo = np.zeros((500, grid_x, grid_y, 8)) - 1
grid_rs = np.zeros((500, grid_x, grid_y, 6)) - 1
grid_hydro = np.ones((500, grid_x, grid_y, 4)) * -9999
pbar = tqdm(total=500, desc='Loading')
for i in range(500):
    pbar.update(1)
    grid_rs[i, :, :, :] = np.load(os.path.join(f'{region_name}_30m_RS', f'{region_name}_30m_RS_{i}.npy'))
    grid_mo[i, :, :, :] = np.load(os.path.join(f'{region_name}_30m_MO', f'{region_name}_30m_MO_{i}.npy'))
grid_rs = np.where(grid_rs < 0, -1, grid_rs)
pbar.close()

# Run BERTH
pbar = tqdm(total=grid_x, desc=f'Running')
for x_id in range(grid_x):
    pbar.update(1)
    for y_id in range(grid_y):
        mo = torch.as_tensor(grid_mo[:, x_id, y_id, [0, 1, 2, 3, 5, 6, 7]], dtype=torch.float).to(device).unsqueeze(0)
        rs = torch.as_tensor(grid_rs[:, x_id, y_id, :], dtype=torch.float).to(device).unsqueeze(0)
        poi_topo = torch.as_tensor(grid_topo[x_id, y_id, :], dtype=torch.float).to(device)
        topo = poi_topo.unsqueeze(0)
        if torch.max(rs) < 0:
            continue
        model_output = model.forward(rs, mo, topo)
        model_hydro = model_output[0].detach().cpu().numpy()
        model_hydro[:, 1] = model_hydro[:, 1] / 10 # soil moisture
        grid_hydro[:, x_id, y_id, :] = model_hydro
pbar.close()

# Export
if not os.path.exists(f'{region_name}_30m_Hydro'):
    os.mkdir(f'{region_name}_30m_Hydro')
for i in range(500):
    np.save(os.path.join(f'{region_name}_30m_Hydro', f'{region_name}_30m_Hydro_{i}.npy'), grid_hydro[i])

In [ ]:
# Visualize
import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib import rcParams, colors as mcolors, cm, ticker as mticker, pyplot as plt
import matplotlib.gridspec as gridspec


et = None
for i in range(0, 365):
    print(i)
    hydro = np.load(f'./Example_Hail_Arabia/DayOffset_{i}.npy')
    if not isinstance(et, np.ndarray):
        et = np.zeros((hydro.shape[0], hydro.shape[1]))
    et += hydro[:, :, 2] + np.abs(np.min(hydro[:, :, 2]))

config = {
    'font.size': 12,
    # 'font.weight': 'bold',
    "mathtext.fontset": 'stix',
    "font.family": 'Microsoft YaHei',
    "axes.unicode_minus": False
}
rcParams.update(config)
fig = plt.figure(figsize=(9, 6), dpi=300)
grid = gridspec.GridSpec(1, 1)

ax = fig.add_subplot(grid[0, 0])
ax.imshow(et, cmap='coolwarm_r', vmin=0, vmax=500)
ax.set_xticks([])
ax.set_yticks([])

axes = fig.get_axes()
axins = inset_axes(axes[0], width="100%", height="50%", loc='lower center',
               bbox_to_anchor=(0, -0.1, 1, 0.1), bbox_transform=axes[0].transAxes)
cbar = plt.colorbar(cm.ScalarMappable(cmap="coolwarm_r", norm=mcolors.Normalize(vmin=0, vmax=500)),
                    orientation='horizontal', label=r'Total ET（mm·y${^{-1}}$）', cax=axins)

plt.subplots_adjust(left=0.05,bottom=0.05,top=0.99,right=0.8,hspace=0.05, wspace=0.05)
plt.show()
